In [137]:
import pandas as pd
import numpy as np
import spikeinterface.extractors as se
import spikeinterface.preprocessing as sp
import matplotlib.pyplot as plt
import neurokit2 as nk
import spikeinterface.extractors as se
import glob
import os
import h5py
import numpy as np
from scipy.signal import butter, filtfilt, resample_poly
from scipy.signal import find_peaks
from collections import defaultdict
import math


In [138]:
base_dir = r"D:\BLA_ChR_resp"

BORIS_DIRS = [
    "RI1_day1_boris",
    "RI1_day2_boris",
    "RI2_day1_boris_post",
    "RI2_day2_boris_post"
]

files = []

for folder in BORIS_DIRS:
    full_path = os.path.join(base_dir, folder)
    csvs = glob.glob(os.path.join(full_path, "*.csv"))
    files.extend(csvs)

# Exclude social-agent scoring
files = [f for f in files if not f.endswith("_sa.csv")]

print("Total BORIS files:", len(files))

Total BORIS files: 94


In [139]:
H5_DIRS = {
    # --- RI1 ---
    "RI1_day1": r"D:\BLA_ChR_resp\BLA_ChR_resp_RI1_hc_day1\h5_outputs",
    "RI1_day2": r"D:\BLA_ChR_resp\BLA_ChR_resp_RI1_hc_day2\h5_outputs",

    # --- RI2 post ---
    "RI2_post_d1": r"D:\BLA_ChR_resp\BLA_ChR_resp_RI2_hc_post_d1\h5_outputs",
    "RI2_post_d2": r"D:\BLA_ChR_resp\BLA_ChR_resp_RI2_hc_post_d2\h5_outputs",
}


In [140]:
all_files = []

for folder in H5_DIRS.values():
    for fname in os.listdir(folder):
        if fname.endswith(".h5"):
            all_files.append(os.path.join(folder, fname))

print("Total files:", len(all_files))


Total files: 94


Helper functions 

In [141]:
def load_boris_for_inspection(csv_path):
    df = pd.read_csv(csv_path)

    # --- Standardize timing column names ---
    df = df.rename(columns={
        "Start (s)": "Start",
        "Stop (s)": "Stop",
        "Duration (s)": "Duration"
    })

    # --- Keep only columns needed to LOOK at behavior ---
    keep_cols = [
        "Behavior",
        "Behavior type",
        "Subject",              # BORIS role: subject / agent
        "Start",
        "Stop",
        "Duration",
        "FPS (frame/s)",
        "Image index start",
        "Image index stop",
        "Media file name"
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].copy()

    # --- Light cleaning ---
    df["Behavior"] = df["Behavior"].astype(str).str.strip().str.lower()
    df["Subject"] = df["Subject"].astype(str).str.strip().str.lower()

    # ============================
    # ADD METADATA FROM FILENAME
    # ============================
    fname = os.path.basename(csv_path)
    parts = fname.replace(".csv", "").split("_")

    # Subject ID = cage_mouse (e.g. 1_2)
    cage = parts[0]
    mouse = parts[1]
    subject_id = f"{cage}_{mouse}"

    # Laser condition
    laser = "ON" if "on" in parts else "OFF"

    df["SubjectID"] = subject_id
    df["Laser"] = laser

    return df

In [206]:
dfs = []

for f in files:
    df = load_boris_for_inspection(f)
    df["File"] = os.path.basename(f).lower()  # lowercase for safety
    dfs.append(df)

boris_df = pd.concat(dfs, ignore_index=True)

print(boris_df.shape)

(3237, 13)


In [207]:
dfs = []
for f in files:
    df = load_boris_for_inspection(f)
    df["File"] = os.path.basename(f)
    dfs.append(df)

boris_df = pd.concat(dfs, ignore_index=True)

In [208]:
print("Total rows:", len(boris_df))
print("Unique sessions:", boris_df["File"].nunique())
print("Unique subjects:", boris_df["SubjectID"].nunique())
print(boris_df.groupby("Laser").size())

Total rows: 3237
Unique sessions: 94
Unique subjects: 24
Laser
OFF    1622
ON     1615
dtype: int64


In [209]:
boris_df = boris_df.copy()

# Extract cage number from SubjectID (e.g. "3_2" → 3)
boris_df["Cage"] = boris_df["SubjectID"].str.split("_").str[0].astype(int)

# Assign group
boris_df["Group"] = boris_df["Cage"].apply(
    lambda x: "GFP" if x % 2 == 0 else "ChR-YFP"
)

In [204]:
print(boris_df.groupby(["Group","Laser"]).size())
print(boris_df.groupby("Group")["SubjectID"].nunique())

Group    Laser
ChR-YFP  OFF      825
         ON       847
GFP      OFF      797
         ON       768
dtype: int64
Group
ChR-YFP    13
GFP        11
Name: SubjectID, dtype: int64


In [211]:
boris_df.columns

Index(['Behavior', 'Behavior type', 'Subject', 'Start', 'Stop', 'Duration',
       'FPS (frame/s)', 'Image index start', 'Image index stop',
       'Media file name', 'SubjectID', 'Laser', 'File', 'Cage', 'Group'],
      dtype='object')

In [212]:
beh_df = boris_df[
    boris_df["Subject"] == "subject"
].copy()

In [216]:
beh_df["Condition"] = beh_df["File"].str.split("_").str[2].str.upper()

In [177]:
histo_bad = {"1_3", "3_3"}

In [213]:
histo_bad = {"1_3", "3_3"}

boris_behavior = boris_df[
    ~boris_df["SubjectID"].isin(histo_bad)
].copy()

print("Behavior subjects:",
      sorted(boris_behavior["SubjectID"].unique()))

Behavior subjects: ['1_1', '1_2', '2_1', '2_2', '2_3', '3_1', '3_2', '4_1', '4_2', '4_3', '5_1', '5_2', '5_3', '6_2', '6_3', '7_1', '7_2', '7_3', '8_1', '8_2', '8_3', '9_1']


In [217]:
social_investigation = [
    "facial sniffing",
    "body sniffing",
    "anogenital sniffing"
]

In [218]:
# First: sum sniff durations per session
social_sum = (
    beh_df[beh_df["Behavior"].isin(sniff_behaviors)]
    .groupby(["SubjectID", "Group", "Condition", "Laser"])["Duration"]
    .sum()
    .reset_index()
    .rename(columns={"Duration": "SocialInvestigationTime"})
)